## 授權與來源（License & Attribution）

本課程自編與改編內容：Copyright 2026 leonjyentub；授權：Apache License 2.0（`../LICENSE`）。
完整來源及授權資訊：`../SOURCES.md`、`../THIRD_PARTY_NOTICES.md`。

參考與改編來源：Aurélien Géron, `ageron/handson-mlp`（Apache-2.0），固定版本 `47eba45aacc85feae51ba7db68dd1ca66cb25e0a`；`leonjyentub/MachineLearning2025` 為課程作者自有專案。
本教學檔案的變更與取捨：參考 handson-mlp Ch.3／Ch.4；另整合 MachineLearning2025 的 Iris、ROC 示範。 將原範例重新編排，配合課堂增補解說、資料流程或縮小實驗；原作者不為本課程背書。

本授權只適用於本 Notebook 中權利人有權授權的程式與文字；第三方資料、圖片與其他權利依各原始條款。

# 03｜分類與模型評估

對應 `slides/03_分類與模型評估.md`。原分冊依教學順序合併，各節仍可獨立選講。

**Colab 示範**：直接上傳此 `.ipynb`，從上到下執行；第一個程式格會安裝缺少的套件，所需資料在使用時自動下載。NumPy 與 scikit-learn 實驗使用 CPU；這些程式沒有可直接切換的 CUDA 後端。輸出存於目前工作目錄。


## Iris 邏輯斯迴歸與多類別

原始分冊：`05_Iris邏輯斯迴歸與多類別.ipynb`。內容與已執行輸出完整併入本課同名 Notebook。

In [ ]:
# 獨立執行：Colab 上傳此檔後，從這一格依序執行；無需掛載 Git 專案。
import importlib.util, subprocess, sys
_needed = {"numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.0,<3", "matplotlib": "matplotlib>=3.7,<4", "scipy": "scipy>=1.11,<2", "sklearn": "scikit-learn>=1.4,<2", "joblib": "joblib>=1.3,<2", "cloudpickle": "cloudpickle>=3,<4", "ipywidgets": "ipywidgets>=8,<9"}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

from pathlib import Path
import hashlib, io, json, os, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# 本機在 programs/ 執行時沿用 data/、outputs/；Colab 則使用自己的工作目錄。
DATA = ROOT / ("data" if (ROOT / "pyproject.toml").exists() else "mlcourse_data")
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
SEED = 42
FAST = os.environ.get("MLCOURSE_PROFILE", "fast") != "full"
_SOURCE_COMMIT = "9e29abbbea6a3ff7250dea64c43c1d6c723095d7"
_SOURCES = {
    "housing.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/housing.tgz", "2364609dc48bec7df3ba9dbb7041478e704ecddcee70ef1827ec3fc49d22c0cc"),
    "lifesat.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/lifesat/lifesat.csv", "e247f7f6c19b0d8c55fb2af293f4fb3b710e31476534ad8e9b7e06ea546a2099"),
}

def setup():
    plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                         "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})
    OUT.mkdir(parents=True, exist_ok=True)
    return {"profile": "fast" if FAST else "full", "seed": SEED, "data": str(DATA)}

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

def data_path(name):
    if name not in (*_SOURCES, "mnist_784_v1.npz"):
        raise ValueError(f"未知資料檔：{name}")
    DATA.mkdir(parents=True, exist_ok=True)
    path = DATA / name
    if name in _SOURCES:
        url, expected_sha = _SOURCES[name]
        if not path.exists():
            print(f"下載 {name} …")
            with urllib.request.urlopen(url, timeout=180) as response:
                blob = response.read()
            if name == "housing.csv":
                with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as archive:
                    blob = archive.extractfile("housing/housing.csv").read()
            if hashlib.sha256(blob).hexdigest() != expected_sha:
                raise ValueError(f"資料校驗失敗：{name}")
            path.write_bytes(blob)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
            raise ValueError(f"本機資料校驗失敗：{name}")
        info = {"url": url, "sha256": expected_sha}
    else:
        if not path.exists():
            print("從 OpenML 下載 MNIST（首次約需數分鐘）…")
            from sklearn.datasets import fetch_openml
            ds = fetch_openml("mnist_784", version=1, as_frame=False,
                              data_home=DATA / "openml_cache", parser="auto")
            X, y = ds.data.astype(np.uint8), ds.target.astype(np.uint8)
            assert X.shape == (70000, 784) and y.shape == (70000,)
            np.savez_compressed(path, X=X, y=y)
        with np.load(path) as archive:
            assert archive["X"].shape == (70000, 784)
            assert archive["y"].shape == (70000,)
        info = {"url": "https://www.openml.org/d/554",
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    manifest_file = DATA / "manifest.json"
    manifest = json.loads(manifest_file.read_text()) if manifest_file.exists() else {}
    manifest[name] = info
    manifest_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
    return path

def housing_split():
    from sklearn.model_selection import train_test_split
    frame = pd.read_csv(data_path("housing.csv"))
    income = pd.cut(frame.median_income, [0, 1.5, 3, 4.5, 6, np.inf], labels=[1, 2, 3, 4, 5])
    train, test = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=income)
    assert not set(train.index) & set(test.index)
    return train, test

setup()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. 先切分，再標準化

二元範例使用 setosa 與 versicolor 的花瓣長／寬。四個原始欄位中挑選兩欄，是預先指定的教學設定；不根據 test 誤差挑欄。

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, ConfusionMatrixDisplay
from scipy.special import expit
iris=load_iris()
mask=iris.target!=2
X=iris.data[mask,2:4];y=iris.target[mask]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.3,stratify=y,random_state=SEED)
scaler=StandardScaler().fit(X_train)
Z_train=scaler.transform(X_train);Z_test=scaler.transform(X_test)
assert np.allclose(scaler.mean_,X_train.mean(axis=0))
print('Training / test:',len(y_train),len(y_test))

## 2. 用平均 log loss 做梯度下降

$z=w^Tx+b$，$p=\sigma(z)$；以 `logaddexp(0,z) - y*z` 計算 loss，避免直接 `log(0)`。由原版「梯度加總」改為平均後，學習率尺度不同，不能原封不動比較每輪數字。

In [ ]:
class LogisticRegressionGD:
    def __init__(self,eta=.2,n_iter=500,random_state=42):
        self.eta=eta;self.n_iter=n_iter;self.random_state=random_state
    def fit(self,X,y):
        X=np.asarray(X,dtype=float);y=np.asarray(y)
        if not set(np.unique(y)) <= {0,1}:
            raise ValueError('This teaching classifier expects 0/1 targets')
        self.w_=np.random.default_rng(self.random_state).normal(0,.01,X.shape[1]+1)
        self.cost_=[]
        for _ in range(self.n_iter):
            z=self.net_input(X);error=self.activation(z)-y
            self.w_[1:] -= self.eta*(X.T@error)/len(X)
            self.w_[0] -= self.eta*error.mean()
            z=self.net_input(X)
            self.cost_.append(np.mean(np.logaddexp(0,z)-y*z))
        return self
    def net_input(self,X):
        return np.asarray(X)@self.w_[1:]+self.w_[0]
    def activation(self,z):
        return expit(z)
    def predict(self,X):
        return (self.net_input(X)>=0).astype(int)
model=LogisticRegressionGD().fit(Z_train,y_train)
assert np.isfinite(model.cost_).all()
assert np.isfinite(np.logaddexp(0,np.array([-1e4,1e4]))).all()
plt.plot(model.cost_);plt.xlabel('Epoch');plt.ylabel('Mean training log loss')
savefig('05_logistic_loss')

## 補充：分類正確不代表損失相同

對應投影片「訓練損失與分類正確率不同」。固定真實類別為正類（$y=1$），只改變模型輸出機率 $\hat p$，比較 $t=0.5$ 下的分類是否正確與二元交叉熵 $-\ln\hat p$。分類結果相同時，機率估計與損失仍可能差很多。

In [ ]:
p_hat=np.array([.9,.6,.1])
ce_demo=pd.DataFrame({'p_hat':p_hat,
                      'class_at_0.5':np.where(p_hat>=.5,'correct','wrong'),
                      'cross_entropy':-np.log(p_hat)})
assert np.allclose(ce_demo.cross_entropy,[0.105,0.511,2.303],atol=5e-4)
ce_demo

In [ ]:
xx,yy=np.meshgrid(np.linspace(-2,2,160),np.linspace(-2,2,160))
prob=model.activation(model.net_input(np.c_[xx.ravel(),yy.ravel()])).reshape(xx.shape)
plt.contourf(xx,yy,prob,levels=np.linspace(0,1,11),cmap='RdBu_r',alpha=.7)
plt.colorbar(label='P(versicolor), uncalibrated')
plt.contour(xx,yy,prob,levels=[.5],colors='black')
plt.scatter(Z_train[:,0],Z_train[:,1],c=y_train,cmap='RdBu_r',edgecolor='k')
plt.scatter(Z_test[:,0],Z_test[:,1],facecolors='none',edgecolors='lime',s=70,label='test positions')
plt.xlabel('Standardized petal length');plt.ylabel('Standardized petal width');plt.legend()
savefig('05_logistic_boundary')
# 固定設定的最後測試；不再依此調 eta 或 epoch。
print('Test accuracy:',accuracy_score(y_test,model.predict(Z_test)))
print('Test log loss:',log_loss(y_test,model.activation(model.net_input(Z_test))))

容易分開的 Iris 二分類可能出現 100% accuracy；不代表所有花種、所有量測條件都能完美分類。概率數值也未經校準驗證。

## 3. 三類別：手寫 OvR 與 OvO

這是一個獨立、預先指定的三類實驗，使用四個特徵。先分割，縮放器只 fit 三類訓練集。OvR 使用最高 decision score；OvO 多數票平手時取 classes 排序中第一類（明示簡化政策）。正式套件可能採分數打破平手，因此結果未必完全一致。

In [ ]:
from itertools import combinations
X_train3,X_test3,y_train3,y_test3=train_test_split(iris.data,iris.target,test_size=.3,stratify=iris.target,random_state=SEED)
scale3=StandardScaler().fit(X_train3)
Z_train3=scale3.transform(X_train3);Z_test3=scale3.transform(X_test3)
classes=np.unique(y_train3)
ovr=[LogisticRegressionGD(n_iter=800).fit(Z_train3,(y_train3==c).astype(int)) for c in classes]
def predict_ovr(Z):
    return classes[np.column_stack([m.net_input(Z) for m in ovr]).argmax(axis=1)]
ovo=[]
for c1,c2 in combinations(classes,2):
    pair=np.isin(y_train3,[c1,c2])
    binary=LogisticRegressionGD(n_iter=800).fit(Z_train3[pair],(y_train3[pair]==c1).astype(int))
    ovo.append((binary,c1,c2))
def predict_ovo(Z):
    votes=np.zeros((len(Z),len(classes)),dtype=int)
    positions={label:i for i,label in enumerate(classes)}
    for m,c1,c2 in ovo:
        positive=m.predict(Z)==1
        votes[:,positions[c1]]+=positive
        votes[:,positions[c2]]+=~positive
    return classes[votes.argmax(axis=1)],votes
ovo_pred,votes=predict_ovo(Z_test3)
assert np.all(votes.sum(axis=1)==len(ovo))
print('K classes:',len(classes),'OvR models:',len(ovr),'OvO models:',len(ovo))
results={'binary_test_accuracy':accuracy_score(y_test,model.predict(Z_test)),
         'OvR_test_accuracy':accuracy_score(y_test3,predict_ovr(Z_test3)),
         'OvO_test_accuracy':accuracy_score(y_test3,ovo_pred),
         'OvO_vote_ties':int(((votes==votes.max(axis=1,keepdims=True)).sum(axis=1)>1).sum())}
report('iris',results)
pd.Series(results)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,(name,pred) in zip(axes,[('OvR',predict_ovr(Z_test3)),('OvO',ovo_pred)]):
    ConfusionMatrixDisplay.from_predictions(y_test3,pred,display_labels=iris.target_names,ax=ax,colorbar=False)
    ax.set_title(name)
plt.tight_layout();savefig('05_iris_multiclass')

## 練習與參考答案

- 10 類時需多少分類器？**OvR：10；OvO：45。**
- OvR 的三個 sigmoid 輸出是否必然加總為 1？**否，三個獨立二分類器並無此約束。**
- 在 split 前對所有 Iris fit scaler，為何不合適？**test 的均值與標準差參與訓練資料轉換；即使此例分數沒變，流程仍有洩漏。**

## 混淆矩陣、閾值與 ROC

原始分冊：`06_混淆矩陣閾值與ROC.ipynb`。內容與已執行輸出完整併入本課同名 Notebook。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from sklearn.metrics import (confusion_matrix,precision_score,recall_score,f1_score,accuracy_score,
    roc_curve,roc_auc_score,precision_recall_curve,average_precision_score)
y=np.array([1,0,1,1,0,0,1,0])
scores=np.array([.95,.85,.8,.7,.6,.4,.3,.1])
def threshold_row(y,scores,threshold):
    pred=scores>=threshold
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'threshold':threshold,'TP':tp,'FP':fp,'FN':fn,'TN':tn,
            'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred),
            'F1':f1_score(y,pred,zero_division=0),'flagged':int(pred.sum()),'cost_FP_plus_4FN':int(fp+4*fn)}
table=pd.DataFrame([threshold_row(y,scores,t) for t in [.8,.5,.2]])
assert table[['TP','FP','FN','TN']].values.tolist()==[[2,1,2,3],[3,2,1,2],[4,3,0,1]]
assert table.cost_FP_plus_4FN.tolist()==[9,6,3]
assert np.isclose(roc_auc_score(y,scores),.6875)
table

## 1. 降低閾值，不是免費提高 recall

此例每個漏判成本 4、誤報成本 1；每天最多檢查 5 筆。在三個候選中，0.2 成本低卻超出容量；0.5 是可行候選裡成本最低者。這是自編教學政策，不能把成本數字當成真實業務資料。

分母為 0 的 precision 在下方以 0 作計算政策；它代表「沒有預測正例，precision 未定義而以 0 填入」，不是實際觀察的零命中率。

In [ ]:
feasible=table[table.flagged<=5]
chosen=feasible.loc[feasible.cost_FP_plus_4FN.idxmin()]
assert chosen.threshold==.5
thresholds=np.r_[np.inf,np.unique(scores)[::-1]]
curve=pd.DataFrame([threshold_row(y,scores,t) for t in thresholds])
finite=curve[np.isfinite(curve.threshold)]
plt.plot(finite.threshold,finite.precision,'o-',label='precision')
plt.plot(finite.threshold,finite.recall,'o-',label='recall')
plt.xlabel('Threshold (predict positive if score >= threshold)');plt.ylabel('Metric');plt.legend()
savefig('06_threshold_tradeoff')
print('Chosen candidate under capacity=5:',chosen.to_dict())

## 2. 原 ROC 程式的方向問題
沿遞增閾值會從 (1,1) 走向 (0,0)，此時對 FPR 做梯形積分可能為負。改以 `inf → 高分 → 低分` 走完整曲線；同分數一起移動，避免任意打破 ties。

In [ ]:
y20=np.array([0,0,1,1,0,1,0,1,0,1,0,1,1,0,1,0,0,1,0,1])
s1=np.array([.1,.4,.35,.8,.2,.6,.3,.9,.4,.7,.15,.85,.5,.25,.65,.55,.45,.75,.05,.95])
s2=np.array([.45,.55,.52,.28,.51,.49,.46,.54,.53,.47,.50,.52,.48,.51,.49,.53,.47,.54,.46,.55])
def manual_roc(y,score):
    y=np.asarray(y);score=np.asarray(score)
    if not (np.any(y==0) and np.any(y==1)):
        raise ValueError('ROC requires both negative and positive examples')
    thresholds=np.r_[np.inf,np.unique(score)[::-1]]
    points=[]
    for threshold in thresholds:
        pred=score>=threshold
        tp=np.sum(pred & (y==1));fp=np.sum(pred & (y==0))
        points.append((fp/np.sum(y==0),tp/np.sum(y==1)))
    return np.array(points),thresholds
rows=[]
for name,score in [('original model 1',s1),('original model 2',s2)]:
    points,thresholds=manual_roc(y20,score)
    auc=np.trapezoid(points[:,1],points[:,0])
    positive=score[y20==1][:,None];negative=score[y20==0][None,:]
    ranking=np.mean((positive>negative)+.5*(positive==negative))
    assert np.allclose(points[[0,-1]],[[0,0],[1,1]])
    assert np.all(np.diff(points[:,0])>=0)
    assert np.isclose(auc,roc_auc_score(y20,score)) and np.isclose(auc,ranking)
    rows.append({'model':name,'AUC':auc,'pairwise_AUC':ranking,
                 'accuracy_at_0.5':accuracy_score(y20,score>=.5)})
    plt.plot(points[:,0],points[:,1],marker='.',label=f'{name}, AUC={auc:.3f}')
plt.plot([0,1],[0,1],'k--');plt.xlabel('False positive rate');plt.ylabel('True positive rate');plt.legend()
savefig('06_original_roc_corrected')
roc_results=pd.DataFrame(rows)
roc_results

## 3. PR 與 ROC 回答不同問題

PR 的 precision 與盛行率相關；ROC 的 FPR 分母是負例，類別高度不平衡時需要同時看 PR 及實際數量。下圖使用 8 筆示例，因此 PR 的 prevalence 參考值為 0.5。Average Precision（AP）以 recall 增量加權 precision，並非把 PR 點做梯形積分；不把兩者混稱。

In [ ]:
p,r,_=precision_recall_curve(y,scores)
fpr,tpr,_=roc_curve(y,scores)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].step(r,p,where='post',label=f'AP={average_precision_score(y,scores):.3f}')
axes[0].axhline(y.mean(),ls='--',color='grey',label='prevalence')
axes[0].set(xlabel='Recall',ylabel='Precision',ylim=(0,1.05));axes[0].legend()
axes[1].plot(fpr,tpr,'o-',label=f'ROC AUC={roc_auc_score(y,scores):.4f}')
axes[1].plot([0,1],[0,1],'k--');axes[1].set(xlabel='FPR',ylabel='TPR');axes[1].legend()
savefig('06_pr_roc')
report('threshold_metrics',{'toy_auc':roc_auc_score(y,scores),'toy_AP':average_precision_score(y,scores),
                           'selected_threshold':float(chosen.threshold),'original_models':roc_results.to_dict('records')})

## 4. 可選互動：先預測混淆矩陣，再移動滑桿
Jupyter 中可互動；靜態 HTML／未安裝 widget 前端時，使用上方固定表格。只用這 8 筆玩具資料，不涉及真實 test 調參。

In [ ]:
import ipywidgets as widgets
from IPython.display import display
def inspect_threshold(threshold=.5):
    row=threshold_row(y,scores,threshold)
    display(pd.DataFrame([row]))
    fig,ax=plt.subplots(figsize=(5,2))
    ax.scatter(scores,np.zeros(len(y)),c=y,cmap='coolwarm',s=90,vmin=0,vmax=1)
    for i,s in enumerate(scores): ax.annotate(str(i+1),(s,.01))
    ax.axvline(threshold,color='k',ls='--');ax.set(xlim=(0,1),ylim=(-.1,.1),yticks=[],xlabel='Score: blue=0, red=1')
    plt.show()
widgets.interactive(inspect_threshold,threshold=widgets.FloatSlider(value=.5,min=0,max=1,step=.05))

## 練習與參考答案

- 分數相同時 AUC 的排序貢獻為何？**0.5。**
- 同一排序把所有分數加 10，AUC 變嗎？**不變，但固定 0.5 閾值的決策可能完全不同。**
- 要部署「precision ≥ 90%」的閾值，在哪裡決定？**validation 或開發集預測；再以未使用的 test 評估，不能保證部署後仍維持 90%。**

## MNIST 分類與錯誤分析

原始分冊：`07_MNIST分類與錯誤分析.ipynb`。內容與已執行輸出完整併入本課同名 Notebook。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. 70,000 張、每張 28×28

先遵循教材前 60,000／後 10,000 的固定資料界線。fast 的 12,000 張來自前 60,000 分層抽樣，並非改用其他數字資料集。模型只在 fit 部分學習；validation 選閾值後不重新 fit，避免模型重訓改變分數尺度。

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,
    classification_report,roc_auc_score,average_precision_score,precision_recall_curve,roc_curve,ConfusionMatrixDisplay)
with np.load(data_path('mnist_784_v1.npz')) as archive:
    images=archive['X'];labels=archive['y']
assert images.shape==(70000,784)
dev_ids=np.arange(60000)
if FAST:
    dev_ids,_=train_test_split(dev_ids,train_size=12000,stratify=labels[:60000],random_state=SEED)
fit_ids,val_ids=train_test_split(dev_ids,test_size=.25,stratify=labels[dev_ids],random_state=SEED)
test_ids=np.arange(60000,70000)
assert not set(fit_ids)&set(val_ids) and not set(dev_ids)&set(test_ids)
X_fit=images[fit_ids].astype(np.float32)/255
X_val=images[val_ids].astype(np.float32)/255
y_fit=labels[fit_ids];y_val=labels[val_ids]
y_fit5=y_fit==5;y_val5=y_val==5
print('fit / validation / locked test:',len(fit_ids),len(val_ids),len(test_ids))
fig,axes=plt.subplots(2,6,figsize=(9,3))
for ax,idx in zip(axes.ravel(),fit_ids[:12]):
    ax.imshow(images[idx].reshape(28,28),cmap='binary');ax.set_title(str(labels[idx]));ax.axis('off')
savefig('07_mnist_examples')

## 2. 「不是 5」就能有高 accuracy？

`cross_val_predict` 對每筆 fit 樣本只使用沒有看過該筆的模型產生 OOF 預測。三折以同一索引比較 SGD 與森林；縮放器放在 Pipeline，每折重新 fit。

OOF 用於開發集比較；選出最高 AP 的模型後，該 AP 也帶模型選擇偏差，不能充當最終 test 成效。

In [ ]:
folds=list(StratifiedKFold(3,shuffle=True,random_state=SEED).split(X_fit,y_fit5))
binary_models={
    'SGD':make_pipeline(StandardScaler(with_mean=False),SGDClassifier(random_state=SEED,max_iter=1000,tol=1e-3)),
    'RandomForest':RandomForestClassifier(n_estimators=60,max_features='sqrt',n_jobs=2,random_state=SEED)
}
oof_scores={};rows=[]
dummy=cross_val_predict(DummyClassifier(strategy='most_frequent'),X_fit,y_fit5,cv=folds)
print('Always-majority baseline: accuracy=',accuracy_score(y_fit5,dummy),'recall=',recall_score(y_fit5,dummy))
for name,model in binary_models.items():
    method='decision_function' if name=='SGD' else 'predict_proba'
    score=cross_val_predict(model,X_fit,y_fit5,cv=folds,method=method,n_jobs=1)
    if method=='predict_proba': score=score[:,1]
    oof_scores[name]=score
    default_threshold=0 if name=='SGD' else .5
    pred=score>=default_threshold
    rows.append({'model':name,'OOF_accuracy':accuracy_score(y_fit5,pred),'OOF_precision':precision_score(y_fit5,pred),
                 'OOF_recall':recall_score(y_fit5,pred),'OOF_AP':average_precision_score(y_fit5,score),
                 'OOF_ROC_AUC':roc_auc_score(y_fit5,score)})
comparison=pd.DataFrame(rows).set_index('model')
comparison

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name,score in oof_scores.items():
    p,r,_=precision_recall_curve(y_fit5,score)
    fpr,tpr,_=roc_curve(y_fit5,score)
    axes[0].step(r,p,where='post',label=name);axes[1].plot(fpr,tpr,label=name)
axes[0].axhline(y_fit5.mean(),ls='--',color='grey',label='prevalence')
axes[0].set(xlabel='Recall',ylabel='Precision');axes[1].set(xlabel='FPR',ylabel='TPR')
for ax in axes:ax.legend()
savefig('07_binary_oof_curves')

## 3. 獨立 validation 選閾值

政策：在 validation 上 precision ≥ 0.90 的候選中，選 recall 最大的閾值。這是經驗估計；少量正例可使精確率不穩，不能宣稱部署保證。`precision_recall_curve` 最後一點沒有對應 threshold，須去除後才對齊。

In [ ]:
selected_name=comparison.OOF_AP.idxmax()
selected=clone(binary_models[selected_name]).fit(X_fit,y_fit5)
def continuous_score(model,X):
    if hasattr(model,'decision_function'):return model.decision_function(X)
    return model.predict_proba(X)[:,list(model.classes_).index(True)]
val_score=continuous_score(selected,X_val)
p,r,thresholds=precision_recall_curve(y_val5,val_score)
feasible=np.flatnonzero(p[:-1]>=.90)
if len(feasible):
    best_index=feasible[np.argmax(r[:-1][feasible])]
    operating_threshold=float(thresholds[best_index])
else:
    operating_threshold=np.inf
val_pred=val_score>=operating_threshold
print('Selected by OOF AP:',selected_name,'threshold:',operating_threshold)
print('Validation precision / recall:',precision_score(y_val5,val_pred,zero_division=0),recall_score(y_val5,val_pred))
print('Validation TP / predicted positives:',int((val_pred&y_val5).sum()),int(val_pred.sum()))

## 4. 多類別與錯誤分析：仍在開發集進行

固定 SGD 為多類別示範模型，比較固定像素除以 255 與額外 StandardScaler 的 CV accuracy。SGD 對多類別採 OvR。10×10 混淆矩陣列為真實、欄為預測；右圖依真實類別正規化後把對角設零，方便看錯誤流向，列加總因此不再是 1。

In [ ]:
multi_folds=list(StratifiedKFold(3,shuffle=True,random_state=SEED).split(X_fit,y_fit))
plain=SGDClassifier(random_state=SEED,max_iter=1000,tol=1e-3)
plain_scores=cross_val_score(plain,X_fit,y_fit,cv=multi_folds,scoring='accuracy')
multi_model=make_pipeline(StandardScaler(with_mean=False),clone(plain))
multi_oof=cross_val_predict(multi_model,X_fit,y_fit,cv=multi_folds)
print('Without StandardScaler CV accuracy:',plain_scores.mean())
print('With StandardScaler OOF accuracy:',accuracy_score(y_fit,multi_oof))
print(classification_report(y_fit,multi_oof,zero_division=0,digits=3))
cm=confusion_matrix(y_fit,multi_oof,labels=np.arange(10))
normalized=cm/cm.sum(axis=1,keepdims=True)
np.fill_diagonal(normalized,0)
fig,axes=plt.subplots(1,2,figsize=(11,4))
ConfusionMatrixDisplay(cm,display_labels=np.arange(10)).plot(ax=axes[0],colorbar=False)
axes[0].set_title('OOF counts')
img=axes[1].imshow(normalized,cmap='Blues');fig.colorbar(img,ax=axes[1])
axes[1].set(xticks=range(10),yticks=range(10),xlabel='Predicted label',ylabel='True label',title='Row-normalized errors (diagonal=0)')
savefig('07_multiclass_confusion')

In [ ]:
offdiag=cm.copy();np.fill_diagonal(offdiag,0)
true_digit,pred_digit=np.unravel_index(offdiag.argmax(),offdiag.shape)
error_positions=np.flatnonzero((y_fit==true_digit)&(multi_oof==pred_digit))[:8]
fig,axes=plt.subplots(2,4,figsize=(8,4))
for ax in axes.ravel():ax.axis('off')
for ax,pos in zip(axes.ravel(),error_positions):
    ax.imshow(X_fit[pos].reshape(28,28),cmap='binary')
    ax.set_title(f'true {true_digit} -> {pred_digit}')
savefig('07_error_gallery')
print('Most frequent OOF error pair:',int(true_digit),'->',int(pred_digit),'count:',offdiag[true_digit,pred_digit])

`macro` 對每一類等權平均；`weighted` 按各類真實樣本數加權，常見類別可掩蓋少數類別弱點。下一格另報告 `micro` F1；單標籤、涵蓋全部類別時它等於 accuracy，並以 assert 驗證。錯誤圖只能提示待驗證的假設，如平移／筆畫相似；不能看完圖就斷言原因。

## 5. 兩個事先指定任務的最後 test

二元模型與閾值已固定；多類別模型也已固定。保留最後 10,000 張，這裡才讀取其標籤來評估。若再利用 test 調整設計，需要新的獨立評估。

In [ ]:
X_test=images[test_ids].astype(np.float32)/255;y_test=labels[test_ids];y_test5=y_test==5
test_score=continuous_score(selected,X_test);test_pred=test_score>=operating_threshold
multi_model.fit(X_fit,y_fit);multi_test=multi_model.predict(X_test)
final={'profile':'fast' if FAST else 'full','fit_rows':len(X_fit),'validation_rows':len(X_val),'test_rows':len(X_test),
       'binary_model':selected_name,'threshold':operating_threshold,
       'binary_test_accuracy':accuracy_score(y_test5,test_pred),
       'binary_test_precision':precision_score(y_test5,test_pred,zero_division=0),
       'binary_test_recall':recall_score(y_test5,test_pred),
       'binary_test_AP':average_precision_score(y_test5,test_score),
       'binary_test_ROC_AUC':roc_auc_score(y_test5,test_score),
       'binary_test_CM':confusion_matrix(y_test5,test_pred,labels=[False,True]).tolist(),
       'multiclass_test_accuracy':accuracy_score(y_test,multi_test),
       'multiclass_test_macro_F1':f1_score(y_test,multi_test,average='macro'),
       'multiclass_test_weighted_F1':f1_score(y_test,multi_test,average='weighted'),
       'multiclass_test_micro_F1':f1_score(y_test,multi_test,average='micro')}
assert np.isclose(final['multiclass_test_micro_F1'],final['multiclass_test_accuracy'])
report('mnist_final',final)
pd.Series(final)

## 6. 選做：多標籤與單一多類別的差異

每張數字可以同時具有 `large (≥7)` 與 `odd` 兩個標籤。使用前 2,000 筆 fit 資料訓練 k-NN，另在前 300 筆 validation 評估；此段為獨立示範，不報告 test 成效。

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
n_small=2000
Y_multi=np.c_[y_fit[:n_small]>=7,y_fit[:n_small]%2==1]
Y_val_multi=np.c_[y_val[:300]>=7,y_val[:300]%2==1]
knn_multi=KNeighborsClassifier(n_neighbors=3,n_jobs=2).fit(X_fit[:n_small],Y_multi)
label_pred=knn_multi.predict(X_val[:300])
print('Targets shape:',Y_multi.shape,'macro F1 across labels:',f1_score(Y_val_multi,label_pred,average='macro'))
pd.DataFrame({'digit':y_val[:8],'pred_large':label_pred[:8,0],'pred_odd':label_pred[:8,1]})

## 7. 選做：784 輸出的像素去噪分類

沿用教材加 0–99 整數噪音，再以 `KNeighborsClassifier` 一次預測 784 個像素，每個像素是 0–255 類別之一。先轉 int16 再加噪音，避免 uint8 溢位；顯示時使用相同色階。這是多輸出分類示範，不是現代影像去噪模型，也不保證所有圖像的 MSE 都降低。

In [ ]:
rng=np.random.default_rng(SEED)
clean_train=images[fit_ids[:n_small]]
clean_val=images[val_ids[:60]]
noisy_train=clean_train.astype(np.int16)+rng.integers(0,100,clean_train.shape,dtype=np.int16)
noisy_val=clean_val.astype(np.int16)+rng.integers(0,100,clean_val.shape,dtype=np.int16)
knn_pixels=KNeighborsClassifier(n_neighbors=3,n_jobs=2).fit(noisy_train,clean_train)
reconstructed=knn_pixels.predict(noisy_val)
assert reconstructed.shape==(60,784)
raw_mse=np.mean((noisy_val.astype(float)-clean_val)**2)
restored_mse=np.mean((reconstructed.astype(float)-clean_val)**2)
print('Validation pixel MSE, noisy / reconstructed:',raw_mse,restored_mse)
fig,axes=plt.subplots(3,3,figsize=(6,6))
for i in range(3):
    for j,(name,arr) in enumerate([('Clean',clean_val),('Noisy',noisy_val),('Reconstructed',reconstructed)]):
        axes[i,j].imshow(arr[i].reshape(28,28),cmap='binary',vmin=0,vmax=255)
        axes[i,j].set_title(name);axes[i,j].axis('off')
savefig('07_multioutput_denoising')
report('mnist_optional',{'multilabel_validation_macro_F1':f1_score(Y_val_multi,label_pred,average='macro'),
                         'noisy_validation_pixel_MSE':raw_mse,'reconstructed_validation_pixel_MSE':restored_mse})

## 練習與參考答案

- OOF 預測能否用來選模型又當作最終分數？**可以做開發比較，但選擇後需獨立 test 評估。**
- validation precision 達 90%，test 一定達到嗎？**不一定，有限樣本誤差與資料差異都會影響。**
- `[large, odd]` 可以同時為 True 嗎？**7 與 9 可以，故為多標籤。**
- 784 個像素輸出與 10 類數字標籤差在哪裡？**前者每個樣本有 784 個目標，後者每個樣本只有一個類別。**